# WTA Match Predictor - Model Training

## Contents

- Import Data and Packages
- Preprocess Data and Define Features
- Train, Test, and Compare Models

### Import Data and Required Packages

In [20]:
# Basic Import
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 
import seaborn as sns
# Modelling
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor,AdaBoostRegressor
from sklearn.svm import SVR
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.model_selection import RandomizedSearchCV
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
import warnings

### Import CSV Data as DataFrame

In [21]:
df = pd.read_csv('data/stud.csv')
df.head()

,tourney_id,tourney_name,surface,draw_size,tourney_level,tourney_date,match_num,winner_id,winner_seed,winner_entry,...,l_1stIn,l_1stWon,l_2ndWon,l_SvGms,l_bpSaved,l_bpFaced,winner_rank,winner_rank_points,loser_rank,loser_rank_points
0,2023-609,Indian Wells,Hard,96,PM,20230306,286,214544,2.0,NaN,...,58.0,37.0,16.0,13.0,8.0,11.0,2,6100,16,2205
1,2023-609,Indian Wells,Hard,96,PM,20230306,285,216347,1.0,NaN,...,55.0,30.0,4.0,10.0,3.0,8.0,1,10585,36,1303
2,2023-609,Indian Wells,Hard,96,PM,20230306,284,221054,NaN,NaN,...,52.0,37.0,10.0,12.0,5.0,8.0,77,784,13,2246
3,2023-609,Indian Wells,Hard,96,PM,20230306,283,201514,NaN,NaN,...,24.0,14.0,12.0,8.0,0.0,5.0,83,743,43,1200
4,2023-609,Indian Wells,Hard,96,PM,20230306,282,201614,5.0,NaN,...,50.0,34.0,21.0,14.0,1.0,4.0,5,4905,49,1080


### Preprocessing

In [22]:
# First, we want to drop columns that we know will not be useful for our model or understanding.
# Information related to the tournament is not relevant
df.drop(columns=['tourney_id', 'tourney_name', 'surface', 'draw_size', 'tourney_level', 'tourney_date'], axis=1, inplace=True)
# Certain information about players (e.g. height, age, hand) are also irrelevant
df.drop(columns=['winner_hand', 'loser_hand', 'winner_ht', 'loser_ht', 'winner_age', 'loser_age', 'winner_ioc', 'loser_ioc'], axis=1, inplace=True)
# Certain match information will also be irrelevant for predictions
df.drop(columns=['match_num', 'best_of', 'round', 'minutes', 'score'], axis=1, inplace=True)

In [23]:
df.head()

,winner_id,winner_seed,winner_entry,winner_name,loser_id,loser_seed,loser_entry,loser_name,w_ace,w_df,...,l_1stIn,l_1stWon,l_2ndWon,l_SvGms,l_bpSaved,l_bpFaced,winner_rank,winner_rank_points,loser_rank,loser_rank_points
0,214544,2.0,NaN,Aryna Sabalenka,206252,16.0,NaN,Barbora Krejcikova,11.0,4.0,...,58.0,37.0,16.0,13.0,8.0,11.0,2,6100,16,2205
1,216347,1.0,NaN,Iga Swiatek,215370,32.0,NaN,Bianca Andreescu,1.0,0.0,...,55.0,30.0,4.0,10.0,3.0,8.0,1,10585,36,1303
2,221054,NaN,NaN,Emma Raducanu,206242,13.0,NaN,Beatriz Haddad Maia,2.0,4.0,...,52.0,37.0,10.0,12.0,5.0,8.0,77,784,13,2246
3,201514,NaN,NaN,Sorana Cirstea,211095,NaN,NaN,Bernarda Pera,4.0,0.0,...,24.0,14.0,12.0,8.0,0.0,5.0,83,743,43,1200
4,201614,5.0,NaN,Caroline Garcia,220367,30.0,NaN,Leylah Fernandez,11.0,3.0,...,50.0,34.0,21.0,14.0,1.0,4.0,5,4905,49,1080


Something very important to note is that, if we implicitly define matches as already having a winner and loser, our model will always predict the match outcome with 100% accuracy, since the data will always indicate who wins and who loses. To fix this, we need to scramble the columns for winners and losers, so it's not obvious to the model who wins the match. We will also rename columns so that there is no mention of a winner or a loser, and will instead encode winner/loser as a single number, 0 or 1.